In [6]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

/home/aifather/venv-qwen3tts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
torch.cuda.is_available()

True

In [8]:
model_id1= "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
model_id2= "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model_id2= "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
model_id3=  "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"

In [9]:
AudioPATH= "./audio"
RefVoicePATH1 = "./audio/ref_voice/warm_reporter_well_paced_steady_friendly.mp3"
RefVoicePATHWav1 = "./audio/ref_voice/warm_reporter_well_paced_steady_friendly.wav"
RefVoicePATHWav2 =f"{AudioPATH}/ref_voice/alienkevin.wav"
referTxt2= "昨晚我嘗試咗去做一個vegetarian pizza，雖然冇咗肉，但係蕃茄醬好rich，起司又melt得剛剛好！"

In [12]:
!huggingface-cli download Qwen/Qwen3-TTS-12Hz-0.6B-Base --local-dir ./Qwen3-TTS-12Hz-0.6B-Base
!huggingface-cli download Qwen/Qwen3-ASR-0.6B --local-dir ./Qwen3-ASR-0.6B

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 13 files: 100%|███████████████████████| 13/13 [00:00<00:00, 944.52it/s]
/home/aifather/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training/Qwen3-TTS-12Hz-0.6B-Base
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0%|                                 | 0/10 [00:00<?, ?it/s]Downloading 'model.safetensors' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/xGOKKLRSlIhH692hSVvI1-gpoa8=.79d6cbd4c98c7bbffe9db2edac07f56cd6637d0d5944b27f6c2b8353840323ea.incomplete'

model.safetensors:   0%|                            | 0.00/1.88G [00:00<?, ?B/s]Downloading 'preprocessor_config.json' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/PYH5dHjks7Ei0Yd3X0Z8xIwsCNQ=.8f7f07346466d5d494ec0d4969d1c3d0190eed72.incomplete'


preprocessor_config.json: 100%|████████████████| 330/330 [00:00<00:00, 1.39MB/s]


Download complete. Moving file to Qwen3-ASR-0.6B/preproce

In [16]:
RefVoicePATHWav2RefVoicePATHWav2

'./audio/ref_voice/alienkevin.wav'

In [17]:

model = Qwen3TTSModel.from_pretrained(
    model_id1,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2", # for nvidia ampere or above archiect
    attn_implementation= "sdpa",  # no flash attention 2 for v100
)


Fetching 4 files: 100%|████████████████████████| 4/4 [00:00<00:00, 20311.40it/s]


In [21]:
model

In [22]:

# Custom Voice Generate Pre-set 9 personal voice
# %%time
# # only for  CustomVoice model used
# wavs, sr = model.generate_custom_voice(
#     text="其实我真的有发现，我是一个特别善于观察别人情绪的人。",
#     language="Chinese", # Pass `Auto` (or omit) for auto language adaptive; if the target language is known, set it explicitly.
#     speaker="Vivian",
#     # instruct="用特别愤怒的语气说", # Omit if not needed.
#     instruct="用自然的香港粤语语气说",
# )
# sf.write("output_custom_voice.wav", wavs[0], sr)

## Voice Clone

In [23]:
%%time
# for ba
wavs, sr = model.generate_voice_clone(
    text="飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。",
    language="chinese",
    ref_audio=RefVoicePATHWav2,
    ref_text=referTxt2,
)
sf.write("output_voice_clone.wav", wavs[0], sr)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[WARNING] Min value of input waveform signal is -1.0005865097045898
[WARNING] Max value of input waveform signal is 1.0000923871994019
CPU times: user 20.4 s, sys: 382 ms, total: 20.8 s
Wall time: 20.8 s


In [31]:
%%time
!python qwen3_voice_clone_cli.py \
  --reference_audio $"{AudioPATH}/ref_voice/alienkevin.wav" \
  --generate_text "今天天气很好！" \
  --qwen3_asr_model ./Qwen3-ASR-0.6B \
  --qwen3_tts_model ./Qwen3-TTS-12Hz-0.6B-Base \
  --no-flash-attn \
  --language Chinese \
  --temperature 0.8 \
  --top_p 0.95 \
  --output_path chinese_cloned.wav
 

Loading Qwen3-ASR model: ./Qwen3-ASR-0.6B on cuda...
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading Qwen3-TTS model: ./Qwen3-TTS-12Hz-0.6B-Base on cuda...
   → Flash Attention 2: DISABLED (older GPU or --no-flash-attn)
talker_config is None. Initializing talker model with default values
speaker_encoder_config is None. Initializing talker model with default values
code_predictor_config is None. Initializing code_predictor model with default values
code_predictor_config is None. Initializing code_predictor model with default values
encoder_config is None. Initializing encoder with default values
decoder_config is None. Initializing decoder with default values
🔊 Transcribing reference audio with ./Qwen3-ASR-0.6B...
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
✅ Auto-generated ref_text: 昨晚我嘗試咗去做一個 vegetarian pizza，雖然冇咗肉，但係番茄醬好rich，皮絲有melts得剛剛好。
🎤 Language: Chinese
🎙️ C

In [32]:
RefVoicePATHWav2

'./audio/ref_voice/alienkevin.wav'